### Business-ready Gold data
- Semantically ready data that provides business insights
- `zip_service_gap_gold` (one row per ZIP), plus `zip_monthly_trend_gold` and `top_rodent_cuisines_gold`.
- The index separates regulator-confirmed rodent evidence from resident 311 reporting, so a ZIP is not ranked worst just for calling 311 the most.

In [ ]:
%run ./00_config

In [ ]:
use_schema(GOLD)   # default schema for this layer; reads from silver are schema-qualified

#### `zip_service_gap_gold`  (one row per ZIP) — the defensible metric
`service_gap_index = z(rodent_evidence_rate) - z(complaint_intensity)`, standardized over High-reliability ZIPs only. Positive = restaurants show rodent evidence but residents file few complaints (an underserved reporting gap).

In [ ]:
%sql
CREATE OR REPLACE TABLE `03_gold`.zip_service_gap_gold AS
WITH rz AS (
  SELECT zip,
         COUNT(DISTINCT camis)                                      AS restaurant_count,
         COUNT(DISTINCT CASE WHEN ever_rodent THEN camis END)       AS rodent_restaurant_count,
         ANY_VALUE(boro)                                            AS boro_r
  FROM `02_silver`.restaurants_silver
  WHERE zip IS NOT NULL
  GROUP BY zip
),
cz AS (
  SELECT zip,
         COUNT(*)                                                   AS complaint_total,
         SUM(CASE WHEN is_confirmed_sighting THEN 1 ELSE 0 END)     AS confirmed_complaints,
         AVG(CASE WHEN is_closed THEN 1.0 ELSE 0.0 END)            AS closure_rate,
         PERCENTILE(days_to_close, 0.5)                             AS median_days_to_close,
         ANY_VALUE(borough)                                         AS boro_c
  FROM `02_silver`.rodent_complaints_silver
  GROUP BY zip
),
base AS (
  SELECT COALESCE(rz.zip, cz.zip)                     AS zip,
         COALESCE(boro_r, boro_c)                     AS borough,
         COALESCE(restaurant_count, 0)                AS restaurant_count,
         COALESCE(rodent_restaurant_count, 0)         AS rodent_restaurant_count,
         COALESCE(complaint_total, 0)                 AS complaint_total,
         COALESCE(confirmed_complaints, 0)            AS confirmed_complaints,
         closure_rate, median_days_to_close
  FROM rz FULL OUTER JOIN cz ON rz.zip = cz.zip
),
rated AS (
  SELECT *,
    CASE WHEN restaurant_count > 0 THEN rodent_restaurant_count / restaurant_count END AS rodent_evidence_rate,
    CASE WHEN restaurant_count > 0 THEN confirmed_complaints    / restaurant_count END AS complaint_intensity,
    CASE WHEN restaurant_count >= 20 AND complaint_total >= 15 THEN 'High' ELSE 'Low' END AS reliability
  FROM base
),
stats AS (
  SELECT AVG(rodent_evidence_rate) AS m_ev, STDDEV(rodent_evidence_rate) AS s_ev,
         AVG(complaint_intensity)  AS m_ci, STDDEV(complaint_intensity)  AS s_ci
  FROM rated WHERE reliability = 'High'
),
scored AS (
  SELECT r.*,
    (rodent_evidence_rate - m_ev) / NULLIF(s_ev, 0) AS z_evidence,
    (complaint_intensity  - m_ci) / NULLIF(s_ci, 0) AS z_intensity
  FROM rated r CROSS JOIN stats
),
final AS (
  SELECT *, ROUND(z_evidence - z_intensity, 3) AS service_gap_index FROM scored
)
SELECT
  zip, borough, reliability,
  CASE WHEN reliability = 'High' THEN
    DENSE_RANK() OVER (ORDER BY (CASE WHEN reliability = 'High' THEN service_gap_index END) DESC NULLS LAST)
  END AS gap_rank,
  service_gap_index,
  CASE WHEN reliability = 'Low'          THEN 'Insufficient data to rank'
       WHEN service_gap_index >=  0.5     THEN 'High confirmed evidence, low reporting'
       WHEN service_gap_index <= -0.5     THEN 'High reporting, lower confirmed evidence'
       ELSE 'Evidence and reporting roughly aligned' END AS gap_label,
  rodent_evidence_rate, complaint_intensity, z_evidence, z_intensity,
  restaurant_count, rodent_restaurant_count, complaint_total, confirmed_complaints,
  ROUND(closure_rate, 3) AS closure_rate, median_days_to_close,
  current_timestamp AS gold_loaded_at
FROM final
ORDER BY gap_rank NULLS LAST;

In [ ]:
%sql
SELECT COUNT(*) AS zips, COUNT(DISTINCT zip) AS distinct_zips FROM `03_gold`.zip_service_gap_gold;

#### `zip_monthly_trend_gold`

In [ ]:
%sql
CREATE OR REPLACE TABLE `03_gold`.zip_monthly_trend_gold AS
WITH comp_m AS (
  SELECT zip, DATE_TRUNC('MONTH', created_ts) AS month,
         COUNT(*) AS complaints,
         SUM(CASE WHEN is_confirmed_sighting THEN 1 ELSE 0 END) AS confirmed_complaints
  FROM `02_silver`.rodent_complaints_silver
  WHERE created_ts IS NOT NULL
  GROUP BY zip, DATE_TRUNC('MONTH', created_ts)
),
viol_m AS (
  SELECT zip, DATE_TRUNC('MONTH', inspection_ts) AS month,
         COUNT(DISTINCT CASE WHEN is_rodent_violation THEN camis END) AS rodent_restaurants
  FROM `02_silver`.restaurant_violations_silver
  WHERE inspection_ts IS NOT NULL
  GROUP BY zip, DATE_TRUNC('MONTH', inspection_ts)
)
SELECT COALESCE(c.zip, v.zip) AS zip, COALESCE(c.month, v.month) AS month,
       COALESCE(complaints, 0) AS complaints,
       COALESCE(confirmed_complaints, 0) AS confirmed_complaints,
       COALESCE(rodent_restaurants, 0) AS rodent_restaurants,
       current_timestamp AS gold_loaded_at
FROM comp_m c FULL OUTER JOIN viol_m v ON c.zip = v.zip AND c.month = v.month;

#### `top_rodent_cuisines_gold`  (story hook)

In [ ]:
%sql
CREATE OR REPLACE TABLE `03_gold`.top_rodent_cuisines_gold AS
SELECT cuisine_description,
       COUNT(DISTINCT camis) AS restaurant_count,
       SUM(CASE WHEN ever_rodent THEN 1 ELSE 0 END) AS rodent_restaurants,
       ROUND(SUM(CASE WHEN ever_rodent THEN 1 ELSE 0 END) / COUNT(DISTINCT camis), 3) AS rodent_rate,
       current_timestamp AS gold_loaded_at
FROM `02_silver`.restaurants_silver
GROUP BY cuisine_description
HAVING COUNT(DISTINCT camis) >= 50
ORDER BY rodent_rate DESC;

#### Descriptions on the gold tables (Genie + dashboard read these)

In [ ]:
%sql
COMMENT ON TABLE `03_gold`.zip_service_gap_gold IS
  'Service Gap Index, one row per NYC ZIP. Positive index means restaurants show rodent evidence but residents file few 311 complaints (an underserved reporting gap). Use reliability = High for rankings.';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN zip COMMENT '5-digit NYC ZIP code';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN borough COMMENT 'NYC borough';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN reliability COMMENT 'High when the ZIP has at least 20 restaurants and 15 complaints, else Low. Only rank High ZIPs';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN gap_rank COMMENT 'Rank by service_gap_index among High-reliability ZIPs, 1 is the largest gap. Null for Low';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN service_gap_index COMMENT 'z(rodent_evidence_rate) minus z(complaint_intensity). Positive means evidence exceeds reporting';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN gap_label COMMENT 'Plain-English reading of the index for this ZIP';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN rodent_evidence_rate COMMENT 'Share of restaurants in the ZIP with a rodent-evidence violation (04K, 04L, 08A)';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN complaint_intensity COMMENT 'Confirmed 311 rodent complaints per restaurant in the ZIP';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN restaurant_count COMMENT 'Distinct restaurants in the ZIP';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN rodent_restaurant_count COMMENT 'Distinct restaurants in the ZIP with rodent evidence';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN complaint_total COMMENT 'All 311 rodent complaints in the ZIP';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN confirmed_complaints COMMENT '311 complaints that are confirmed sightings';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN closure_rate COMMENT 'Share of the ZIP complaints that are closed';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN median_days_to_close COMMENT 'Median days to close a complaint in the ZIP';
ALTER TABLE `03_gold`.zip_service_gap_gold ALTER COLUMN gold_loaded_at COMMENT 'Lineage: when this gold table was built';

In [ ]:
%sql
COMMENT ON TABLE `03_gold`.zip_monthly_trend_gold IS 'Monthly complaint and rodent-violation counts per ZIP, for trend charts.';
ALTER TABLE `03_gold`.zip_monthly_trend_gold ALTER COLUMN month COMMENT 'Month (first day) of the activity';
ALTER TABLE `03_gold`.zip_monthly_trend_gold ALTER COLUMN complaints COMMENT 'All 311 rodent complaints filed that month in the ZIP';
ALTER TABLE `03_gold`.zip_monthly_trend_gold ALTER COLUMN rodent_restaurants COMMENT 'Distinct restaurants cited for rodent evidence that month';
ALTER TABLE `03_gold`.zip_monthly_trend_gold ALTER COLUMN gold_loaded_at COMMENT 'Lineage: when this gold table was built';
COMMENT ON TABLE `03_gold`.top_rodent_cuisines_gold IS 'Cuisine categories ranked by the share of restaurants ever cited for rodent evidence (min 50 restaurants).';
ALTER TABLE `03_gold`.top_rodent_cuisines_gold ALTER COLUMN rodent_rate COMMENT 'rodent_restaurants divided by restaurant_count';
ALTER TABLE `03_gold`.top_rodent_cuisines_gold ALTER COLUMN gold_loaded_at COMMENT 'Lineage: when this gold table was built';

### Sanity checks: prove the obvious analysis is wrong
Raw complaint volume should NOT line up with confirmed rodent evidence. If these two lists differ, ranking ZIPs by 311 calls measures civic reporting, not rodents.

In [ ]:
%sql
-- near-zero correlation is the whole point
SELECT ROUND(CORR(complaint_total, rodent_evidence_rate), 3) AS corr_rawcalls_vs_evidence
FROM `03_gold`.zip_service_gap_gold WHERE reliability = 'High';

In [ ]:
%sql
-- Top 10 by RAW complaint volume
SELECT zip, borough, complaint_total, rodent_evidence_rate, service_gap_index
FROM `03_gold`.zip_service_gap_gold WHERE reliability = 'High'
ORDER BY complaint_total DESC LIMIT 10;

In [ ]:
%sql
-- Top 10 by SERVICE GAP INDEX (evidence exceeds reporting) — a different list
SELECT zip, borough, service_gap_index, rodent_evidence_rate, complaint_total, gap_label
FROM `03_gold`.zip_service_gap_gold WHERE reliability = 'High'
ORDER BY service_gap_index DESC LIMIT 10;

#### Verify gold layer

In [ ]:
%sql
SHOW TABLES IN `03_gold`;